In [1]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [24]:
from datetime import datetime
from utils.types import TransformerType, EntityType, CatalogueTransformerType
import boto3
from typing import Any
import smart_open
import csv
from pydantic import BaseModel
from concurrent.futures import ThreadPoolExecutor
from itertools import chain
import polars as pl
from utils.aws import df_from_s3_parquet

S3 = boto3.client("s3")
TRANSPORT_PARAMS = {"client": S3}
S3_PARALLELISM = 10

BUCKET_NAME = "wellcomecollection-catalogue-graph"
PIPELINE_DATE = "2025-10-02"


def get_csv_from_s3(s3_uri: str) -> list[Any]:
    with smart_open.open(s3_uri, "r", transport_params=TRANSPORT_PARAMS) as f:
        return list(csv.DictReader(f))


class BaseDebugger(BaseModel):
    bucket_name: str = BUCKET_NAME
    pipeline_date: str = PIPELINE_DATE

    @property
    def service_prefix(self) -> str:
        raise NotImplementedError()

    @property
    def windows_prefix(self) -> str:
        return f"{self.service_prefix}/{self.pipeline_date}/windows"

    def strf_window(self, window: tuple[datetime, datetime]) -> str:
        return f"{window[0].strftime('%Y%m%dT%H%M')}-{window[1].strftime('%Y%m%dT%H%M')}"

    def strp_window(self, window: str) -> tuple[datetime, datetime]:
        raw_start, raw_end = window.split("-")
        window_start = datetime.strptime(raw_start, "%Y%m%dT%H%M")
        window_end = datetime.strptime(raw_end, "%Y%m%dT%H%M")
        return window_start, window_end

    def get_all_incremental_windows(self) -> list[tuple[datetime, datetime]]:
        all_windows = []
        paginator = S3.get_paginator("list_objects_v2")
        prefix = f"{self.windows_prefix}/"
        for page in paginator.paginate(Bucket=self.bucket_name, Prefix=prefix, Delimiter="/"):
            for common_prefix in page["CommonPrefixes"]:
                full_prefix = common_prefix["Prefix"]
                window = full_prefix[len(prefix):].rstrip("/")
                all_windows.append(self.strp_window(window))
        
        return all_windows

    def get_incremental_uri(window: tuple[datetime, datetime]) -> str:
        raise NotImplementedError()
    
    def get_incremental_file(s3_uri: str) -> pl.DataFrame:
        raise NotImplementedError()
    
    def get_incremental_df(self, window: tuple[datetime, datetime]):
        uri = self.get_window_uri(window)
    
        try:
            df = self.get_incremental_file(uri)
        except OSError:
            print(f"File '{uri}' does not exist")
            df = pl.DataFrame()

        if len(df) > 0:
            df = df.with_columns(pl.lit(window[0]).alias("window_start"))
            df = df.with_columns(pl.lit(window[1]).alias("window_end"))
        return df

    def dataframe_from_incremental_files(self):
        all_windows = self.get_all_incremental_windows()
        print(f"There are {len(all_windows):,} {self.service_prefix} time windows associated with pipeline date {self.pipeline_date}.")
        
        with ThreadPoolExecutor(max_workers=S3_PARALLELISM) as executor:
            dfs = executor.map(self.get_incremental_df, all_windows)

        full_df = pl.concat(df for df in dfs if len(df) > 0)
        print(f"Retrieved {len(full_df):,} rows")
    
        return full_df.sort("window_end")        

class BulkLoaderDebugger(BaseDebugger):
    transformer_type: TransformerType
    entity_type: EntityType

    @property
    def service_prefix(self) -> str:
        return "graph_bulk_loader"

    @property
    def file_name(self) -> str:
        return f"{self.transformer_type}__{self.entity_type}.csv"

    @property
    def full_reindex_uri(self) -> str:
        return f"s3://{self.bucket_name}/{self.service_prefix}/{self.pipeline_date}/{self.file_name}"

    def get_window_uri(self, window: tuple[datetime, datetime]) -> str:
        return f"s3://{self.bucket_name}/{self.windows_prefix}/{self.strf_window(window)}/{self.file_name}"

    def get_incremental_file(self, s3_uri: str) -> pl.DataFrame:
        df = pl.DataFrame(get_csv_from_s3(s3_uri))
        df.columns = ["id"]
        return df

    def dataframe_from_full_reindex_file(self):
        df = pl.DataFrame(get_csv_from_s3(self.full_reindex_uri))
        print(f"Retrieved a full reindex bulk load file storing {len(df):,} rows")
        return df


class GraphRemoverIncrementalDebugger(BaseDebugger):
    transformer_type: CatalogueTransformerType
    entity_type: EntityType

    @property
    def service_prefix(self) -> str:
        return "graph_remover_incremental"

    @property
    def file_name(self) -> str:
        return f"{self.transformer_type}__{self.entity_type}.parquet"

    def get_window_uri(self, window: tuple[datetime, datetime]) -> str:
        return f"s3://{self.bucket_name}/{self.windows_prefix}/{self.strf_window(window)}/deleted_ids/{self.file_name}"

    def get_incremental_file(self, s3_uri: str):
        return df_from_s3_parquet(s3_uri)

In [25]:
debugger = BulkLoaderDebugger(
    transformer_type="catalogue_work_identifiers",
    entity_type="nodes"
)

df = debugger.dataframe_from_incremental_files()
#df = debugger.dataframe_from_full_reindex_file()

There are graph_bulk_loader 15,186 time windows associated with pipeline date 2025-10-02.
File 's3://wellcomecollection-catalogue-graph/graph_bulk_loader/2025-10-02/windows/20260218T1550-20260218T1600/catalogue_work_identifiers__nodes.csv' does not exist
Retrieved 899,677 rows


In [28]:
df.filter(pl.col(":ID") == "GALTON/2/9/6")

:ID,:LABEL,id:String,label:String,window_start,window_end
str,str,str,str,datetime[μs],datetime[μs]


In [29]:
debugger = GraphRemoverIncrementalDebugger(
    transformer_type="catalogue_work_identifiers",
    entity_type="nodes"
)

df = debugger.dataframe_from_incremental_files()

There are graph_remover_incremental 14,468 time windows associated with pipeline date 2025-10-02.
Retrieved 1,117 rows


In [35]:
df.filter(pl.col("id") == "GALTON/2/9/6")

column_0,window_start,window_end
str,datetime[μs],datetime[μs]
